In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully!")

DATA_PATH = Path("../data/processed/rentals_investigation.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Libraries imported successfully!
Dataset loaded successfully!
Shape: (17890, 17)


,url,listing_type,name,price,category,bedrooms,bathrooms,floor_area,location,condition,amenities,region,locality,parking_space,is_furnished,lat,lng
0,https://tonaton.com/a-MfYFNdCgCTytOcmvZIh0Kzrj...,rentals,"2bdrm Apartment in Charclem Ventures, East Leg...",9196,Flats,2.0,2.0,145.0,"Greater Accra, East Legon",Used,"24-hour Electricity,Air Conditioning,Balcony,C...",Greater Accra,East Legon,False,Semi-Furnished,5.635828,-0.161359
1,https://tonaton.com/a-MfYFNdCgwtb8Itol9odpMPvC...,rentals,"2bdrm Apartment in , Dzorwulu for rent",7500,Flats,2.0,2.0,100.0,"Greater Accra, Dzorwulu",Used,"24-hour Electricity,Air Conditioning,Apartment...",Greater Accra,Dzorwulu,False,Unfurnished,5.614123,-0.195647
2,https://tonaton.com/a-MfYFNdCgPjnzPYwNFEv5JrGy...,rentals,"4bdrm House in Lakeside Estate, for Rent",11200,Detached,4.0,5.0,100.0,"Greater Accra, Ashaley Botwe, Lake Side Estate",New,"Tiled Floor,Chandelier,Dining Area,Balcony,Pop...",Greater Accra,Ashaley Botwe,False,Semi-Furnished,5.680213,-0.137248
3,https://tonaton.com/a-MfYFNdCgwmXHQRNJiSR3rok8...,rentals,"3bdrm House in Ayathar Bedroom, Teshie for Rent",2500,Detached,3.0,3.0,17.0,"Greater Accra, Teshie",Used,"Pop Ceiling,Pre-Paid Meter,24-hour Electricity...",Greater Accra,Teshie,False,Semi-Furnished,5.583166,-0.104608
4,https://tonaton.com/a-MfYFNdCgn3Bcs511lKnFE19i...,rentals,"3bdrm House in Manet Viile Estate, Spintex for...",9146,Detached,3.0,3.0,160.0,"Greater Accra, Spintex",Used,"Pre-Paid Meter,Dining Area,Chandelier,Balcony,...",Greater Accra,Spintex,False,Semi-Furnished,5.637614,-0.126606


In [12]:
# Check that the target column exists
if "price" not in df.columns:
    raise ValueError("The dataset does not contain a 'price' column.")

# Remove columns that are not useful as model features
df = df.drop(columns=["url", "name"], errors="ignore")

# Separate input features and target
X = df.drop(columns=["price"]).copy()
y = df["price"].copy()

print("Features and target separated successfully!")
print("X shape:", X.shape)
print("y shape:", y.shape)

Features and target separated successfully!
X shape: (17890, 14)
y shape: (17890,)


In [13]:
def count_amenities(value):
    if pd.isna(value) or str(value).strip() == "":
        return 0

    return len([
        item for item in str(value).split(",")
        if item.strip()
    ])


X["amenities_count"] = X["amenities"].apply(count_amenities)

X[["amenities", "amenities_count"]].head()

,amenities,amenities_count
0,"24-hour Electricity,Air Conditioning,Balcony,C...",14
1,"24-hour Electricity,Air Conditioning,Apartment...",11
2,"Tiled Floor,Chandelier,Dining Area,Balcony,Pop...",16
3,"Pop Ceiling,Pre-Paid Meter,24-hour Electricity...",6
4,"Pre-Paid Meter,Dining Area,Chandelier,Balcony,...",14


In [14]:
X = X.drop(columns=["amenities"])

print("Original amenities column removed.")
print("New feature shape:", X.shape)

Original amenities column removed.
New feature shape: (17890, 14)


In [15]:
if "parking_space" in X.columns:
    X["parking_space"] = X["parking_space"].astype(int)

print(X["parking_space"].value_counts())

parking_space
0    17875
1       15
Name: count, dtype: int64


In [16]:
numerical_features = [
    col for col in X.columns
    if pd.api.types.is_numeric_dtype(X[col])
]

categorical_features = [
    col for col in X.columns
    if (
        pd.api.types.is_object_dtype(X[col])
        or pd.api.types.is_string_dtype(X[col])
        or isinstance(X[col].dtype, pd.CategoricalDtype)
    )
]

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['bedrooms', 'bathrooms', 'floor_area', 'parking_space', 'lat', 'lng', 'amenities_count']

Categorical features:
['listing_type', 'category', 'location', 'condition', 'region', 'locality', 'is_furnished']


In [17]:
print("X exists:", "X" in globals())
print("Numerical features exist:", "numerical_features" in globals())
print("Categorical features exist:", "categorical_features" in globals())

print("\nFeature dataset shape:", X.shape)
print("\nMissing values:")
print(X.isnull().sum())

X.head()

X exists: True
Numerical features exist: True
Categorical features exist: True

Feature dataset shape: (17890, 14)

Missing values:
listing_type       0
category           0
bedrooms           0
bathrooms          0
floor_area         0
location           0
condition          0
region             0
locality           0
parking_space      0
is_furnished       0
lat                0
lng                0
amenities_count    0
dtype: int64


,listing_type,category,bedrooms,bathrooms,floor_area,location,condition,region,locality,parking_space,is_furnished,lat,lng,amenities_count
0,rentals,Flats,2.0,2.0,145.0,"Greater Accra, East Legon",Used,Greater Accra,East Legon,0,Semi-Furnished,5.635828,-0.161359,14
1,rentals,Flats,2.0,2.0,100.0,"Greater Accra, Dzorwulu",Used,Greater Accra,Dzorwulu,0,Unfurnished,5.614123,-0.195647,11
2,rentals,Detached,4.0,5.0,100.0,"Greater Accra, Ashaley Botwe, Lake Side Estate",New,Greater Accra,Ashaley Botwe,0,Semi-Furnished,5.680213,-0.137248,16
3,rentals,Detached,3.0,3.0,17.0,"Greater Accra, Teshie",Used,Greater Accra,Teshie,0,Semi-Furnished,5.583166,-0.104608,6
4,rentals,Detached,3.0,3.0,160.0,"Greater Accra, Spintex",Used,Greater Accra,Spintex,0,Semi-Furnished,5.637614,-0.126606,14


In [20]:
engineered_df = X.copy()

engineered_df["price"] = y

print("Engineered dataset shape:", engineered_df.shape)

engineered_df.head()

Engineered dataset shape: (17890, 15)


,listing_type,category,bedrooms,bathrooms,floor_area,location,condition,region,locality,parking_space,is_furnished,lat,lng,amenities_count,price
0,rentals,Flats,2.0,2.0,145.0,"Greater Accra, East Legon",Used,Greater Accra,East Legon,0,Semi-Furnished,5.635828,-0.161359,14,9196
1,rentals,Flats,2.0,2.0,100.0,"Greater Accra, Dzorwulu",Used,Greater Accra,Dzorwulu,0,Unfurnished,5.614123,-0.195647,11,7500
2,rentals,Detached,4.0,5.0,100.0,"Greater Accra, Ashaley Botwe, Lake Side Estate",New,Greater Accra,Ashaley Botwe,0,Semi-Furnished,5.680213,-0.137248,16,11200
3,rentals,Detached,3.0,3.0,17.0,"Greater Accra, Teshie",Used,Greater Accra,Teshie,0,Semi-Furnished,5.583166,-0.104608,6,2500
4,rentals,Detached,3.0,3.0,160.0,"Greater Accra, Spintex",Used,Greater Accra,Spintex,0,Semi-Furnished,5.637614,-0.126606,14,9146


In [21]:
OUTPUT_PATH = Path("../data/processed/rentals_engineered.csv")

# Ensure the output directory exists
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save the engineered dataset
engineered_df.to_csv(OUTPUT_PATH, index=False)

print("Feature engineering completed successfully!")
print("Saved to:", OUTPUT_PATH)
print("Final dataset shape:", engineered_df.shape)

Feature engineering completed successfully!
Saved to: ..\data\processed\rentals_engineered.csv
Final dataset shape: (17890, 15)
